# Phase 1 fine-tune: **Weighted-Sum** (layers 8-9-10 · int8 · RAM) — "1-99"

After the diagnosis that layer 9 alone is limiting, this **enriches** the feature with a **learnable weighted sum** of layers 8, 9 and 10. The probing peak sits in that band, and 11 was dropped because it drifts toward the pretraining objective and is weak for ASR.

**The "1-99" method:** the backbone runs **once** and the features are written to **RAM** as **per-channel int8** (about 44 GB). There is **no disk feature cache**. Epochs 2 to 100 read from RAM without touching the disk, so steady state is a matter of seconds.

- Only the **model output** is written to disk (`/content/drive/MyDrive/CLEAR/phase1_ws`).
- **RAM budget:** about 40 GB for train plus about 4 GB for dev, in int8. An L4 high-mem instance (about 50 GB) is **right at the edge**. On OOM, cut `WS_LAYERS` down to `[9,10]` or limit `MAX_TRAIN_SAMPLES`.

Baseline reference: **val-CER 8.2%**.

## 1. Drive and dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q "transformers>=4.44" "datasets>=2.20" jiwer torchaudio soundfile

## 2. Import & config
`WS_LAYERS` is the set of layers entering the weighted sum. The paths are kept separate from the baseline and from Phase 2.

In [ ]:
import os, json, time, gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from datasets import load_dataset, Audio
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, HubertModel
import jiwer

# ================= CONFIG =================
BACKBONE_ID   = "utter-project/mHuBERT-147"
DATASET_ID    = "openslr/librispeech_asr"
SR            = 16_000
HID           = 768

# --- WEIGHTED-SUM (3 layers · int8 · RAM-resident) ---
WS_LAYERS     = [8, 9, 10]        # probing peak is 8-10; 11 (pretraining target) dropped
N_WS          = len(WS_LAYERS)
CALIB_N       = 512               # sample count for the int8 per-channel scale calibration

# --- PATHS (model output ONLY, features are NEVER written to disk) ---
OUTPUT_DIR    = "/content/drive/MyDrive/CLEAR/phase1_ws"

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES  = None

PRECOMPUTE_BATCH  = 12
TRAIN_BATCH       = 512
NUM_EPOCHS        = 100
LR                = 2e-3
LR_PATIENCE       = 4
LR_FACTOR         = 0.5
STOP_PATIENCE     = 12            # early stopping
NUM_WORKERS       = 0             # the features live in RAM, no I/O, so workers are pointless (and no COW risk)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[ENV] torch={torch.__version__} device={DEVICE}"
      + (f" gpu={torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Vocab and tokenizer (byte-for-byte identical to the baseline)

In [4]:
chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"]     = len(vocab)
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)
with open("vocab.json", "w") as f:
    json.dump(vocab, f)
tokenizer = Wav2Vec2CTCTokenizer("vocab.json", unk_token="[UNK]",
                                 pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SR, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)
VOCAB_SIZE = len(vocab)
BLANK_ID   = vocab["[PAD]"]
UNK_ID     = vocab["[UNK]"]
ID2CH      = {i: c for c, i in vocab.items()}
print(f"[VOCAB] {VOCAB_SIZE} token, blank={BLANK_ID}")

[VOCAB] 30 token, blank=29


## 4. "1-99" precompute — backbone 1×, int8 per-channel, into RAM
The backbone processes all the data once. `hidden_states[8,9,10]` are stacked, quantised to int8 with a **per-channel scale** derived from a small calibration set, and kept in RAM. Nothing is written to disk.

In [ ]:
# ============================================================
# The "1-99" method: the backbone runs ONCE and the features are written to RAM as int8.
# Epochs 2-100 read from RAM without touching the disk -> steady state in seconds.
# Per-channel int8 quantisation: the [N_WS,768] scale comes from a small calibration set.
# ============================================================

class RawAudioDataset(Dataset):
    def __init__(self, hf_ds): self.ds = hf_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        ex = self.ds[i]
        return np.asarray(ex["audio"]["array"], dtype=np.float32), ex["text"].upper()

def raw_collate(items):
    return [it[0] for it in items], [it[1] for it in items]

def gpu_normalize(iv, lengths):
    for i in range(iv.shape[0]):
        L = lengths[i]; x = iv[i, :L]
        iv[i, :L] = (x - x.mean()) / torch.sqrt(x.var(unbiased=False) + 1e-5)
    return iv

@torch.no_grad()
def _forward_batch(backbone, buf_audio):
    Ls = [len(a) for a in buf_audio]; Tmax = max(Ls)
    batch = np.zeros((len(buf_audio), Tmax), dtype=np.float32)
    for i, a in enumerate(buf_audio): batch[i, :len(a)] = a
    iv = torch.from_numpy(batch).pin_memory().to(DEVICE, non_blocking=True)
    iv = gpu_normalize(iv, Ls).to(torch.float16)
    am = torch.zeros((len(buf_audio), Tmax), dtype=torch.long, device=DEVICE)
    for i, L in enumerate(Ls): am[i, :L] = 1
    out = backbone(iv, attention_mask=am, output_hidden_states=True)
    hs = torch.stack([out.hidden_states[L] for L in WS_LAYERS], dim=1)   # [B,N_WS,T,768] fp16
    valid = backbone._get_feat_extract_output_lengths(am.sum(-1)).tolist()
    return hs.float().cpu().numpy(), [int(v) for v in valid]

def _load_split(split, max_samples):
    if split == "train.100":
        hf = load_dataset(DATASET_ID, data_files={"train": "clean/train.100/*.parquet"},
                          split="train", verification_mode="no_checks")
    else:
        hf = load_dataset(DATASET_ID, data_files={"validation": "clean/validation/*.parquet"},
                          split="validation", verification_mode="no_checks")
    hf = hf.cast_column("audio", Audio(sampling_rate=SR))
    if max_samples: hf = hf.select(range(min(max_samples, len(hf))))
    return hf

@torch.no_grad()
def build_ram_cache(backbone, split, tag, scale=None, max_samples=None):
    hf = _load_split(split, max_samples)
    # sort by length, which cuts the padding waste in the backbone batches
    raw_lens = [len(hf[i]["audio"]["array"]) for i in range(len(hf))]
    order = sorted(range(len(raw_lens)), key=lambda i: raw_lens[i])
    hf = hf.select(order); raw_lens = [raw_lens[i] for i in order]
    feat_lens = [int(v) for v in
                 backbone._get_feat_extract_output_lengths(torch.tensor(raw_lens)).tolist()]
    total = sum(feat_lens); gb = total * N_WS * HID / 1e9
    print(f"[RAM] {tag}: {len(hf)} samples, {total:,} frames -> allocating {gb:.1f} GB int8...")
    feats = np.empty((total, N_WS, HID), dtype=np.int8)      # << LARGE RAM ALLOC

    loader = DataLoader(RawAudioDataset(hf), batch_size=PRECOMPUTE_BATCH,
                        num_workers=8, collate_fn=raw_collate, prefetch_factor=4)

    # --- 1) CALIBRATION: per-channel abs-max over the first CALIB_N samples -> scale ---
    if scale is None:
        absmax = np.zeros((N_WS, HID), np.float32); seen = 0
        for buf_audio, _ in loader:
            hs, valid = _forward_batch(backbone, buf_audio)
            for i, T in enumerate(valid):
                absmax = np.maximum(absmax, np.abs(hs[i, :, :T, :]).max(axis=1))
                seen += 1
            if seen >= CALIB_N: break
        scale = (absmax / 127.0); scale[scale < 1e-8] = 1e-8
        print(f"[RAM] {tag}: scale ready (per-channel, {seen} samples), median={np.median(scale):.4g}")

    # --- 2) MAIN PASS: quantise all the data to int8 and write it to RAM ---
    labels, texts = [], []
    wp, done, t0 = 0, 0, time.perf_counter()
    for buf_audio, buf_text in loader:
        hs, valid = _forward_batch(backbone, buf_audio)
        for i, (T, txt) in enumerate(zip(valid, buf_text)):
            block = hs[i, :, :T, :].transpose(1, 0, 2)                    # [T,N_WS,768]
            feats[wp:wp+T] = np.clip(np.round(block / scale), -127, 127).astype(np.int8)
            wp += T
            labels.append(tokenizer(txt).input_ids); texts.append(txt); done += 1
        if done % 2000 < PRECOMPUTE_BATCH:
            print(f"[RAM] {tag}: {done}/{len(hf)} | {done/(time.perf_counter()-t0):.0f} samples/s")
    assert wp == total, (wp, total)
    offsets = np.concatenate([[0], np.cumsum(feat_lens)])[:-1].astype(np.int64)
    print(f"[RAM] {tag}: done - {done} samples, {gb:.1f} GB, {(time.perf_counter()-t0)/60:.1f} min")
    return {"feats": feats, "offsets": offsets, "lengths": feat_lens,
            "labels": labels, "texts": texts}, scale

In [ ]:
# load the backbone -> build the train and dev RAM cache (same scale) -> release the backbone
print(f"[PRE] loading the backbone ({BACKBONE_ID}, fp16)...")
backbone = HubertModel.from_pretrained(BACKBONE_ID, torch_dtype=torch.float16).to(DEVICE).eval()

train_cache, FEAT_SCALE = build_ram_cache(backbone, "train.100", "train",
                                          scale=None, max_samples=MAX_TRAIN_SAMPLES)
dev_cache,   _          = build_ram_cache(backbone, "validation", "dev",
                                          scale=FEAT_SCALE, max_samples=MAX_EVAL_SAMPLES)

del backbone; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"[PRE] backbone released. RAM cache ready "
      f"(train={len(train_cache['lengths'])}, dev={len(dev_cache['lengths'])}).")

## 5. RAM dataset (int8, 3 layers) + sampler + collate

In [ ]:
class RamFeatDataset(Dataset):
    """The int8 features sit in one contiguous RAM array. __getitems__ returns views (no copy);
    dequantisation happens inside the model on the GPU. It runs with num_workers=0, so there is no COW or pickle trouble."""
    def __init__(self, cache):
        self.feats   = cache["feats"]        # np.int8 [total, N_WS, 768] (RAM)
        self.offsets = cache["offsets"]
        self.lengths = cache["lengths"]
        self.labels  = cache["labels"]
        self.texts   = cache["texts"]
    def __len__(self): return len(self.lengths)
    def __getitem__(self, i):
        o, L = int(self.offsets[i]), self.lengths[i]
        return (torch.from_numpy(self.feats[o:o+L]),
                torch.tensor(self.labels[i], dtype=torch.long), L, i)
    def __getitems__(self, indices):
        out = []
        for idx in indices:
            o, L = int(self.offsets[idx]), self.lengths[idx]
            out.append((torch.from_numpy(self.feats[o:o+L]),
                        torch.tensor(self.labels[idx], dtype=torch.long), L, idx))
        return out

class BucketBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = np.array(lengths); self.batch_size = batch_size; self.shuffle = shuffle
    def __iter__(self):
        if self.shuffle:
            noise = np.random.rand(len(self.lengths))
            order = np.lexsort((noise, self.lengths))
            batches = [order[i:i+self.batch_size].tolist()
                       for i in range(0, len(order), self.batch_size)]
            BIN_SIZE = 8
            for i in range(0, len(batches), BIN_SIZE):
                b = batches[i:i+BIN_SIZE]; np.random.shuffle(b); batches[i:i+BIN_SIZE] = b
        else:
            order = np.argsort(self.lengths)
            batches = [order[i:i+self.batch_size].tolist()
                       for i in range(0, len(order), self.batch_size)]
        yield from batches
    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

def collate(items):
    feats, labs, lens, idxs = zip(*items)
    B = len(feats); Smax = max(len(l) for l in labs)
    x = torch.nn.utils.rnn.pad_sequence(feats, batch_first=True)      # int8, pad=0
    y = torch.full((B, Smax), BLANK_ID, dtype=torch.long)
    for i, l in enumerate(labs): y[i, :len(l)] = l
    return (x, y, torch.tensor(lens, dtype=torch.long),
            torch.tensor([len(l) for l in labs], dtype=torch.long), list(idxs))

train_ds = RamFeatDataset(train_cache)
dev_ds   = RamFeatDataset(dev_cache)
train_dl = DataLoader(train_ds, batch_sampler=BucketBatchSampler(train_ds.lengths, TRAIN_BATCH),
                      collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=True)
dev_dl   = DataLoader(dev_ds, batch_sampler=BucketBatchSampler(dev_ds.lengths, TRAIN_BATCH, shuffle=False),
                      collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=True)
print(f"[DATA] train={len(train_ds)} dev={len(dev_ds)} | {len(train_dl)} train batch/epoch")

## 6. Weighted-Sum head + decode + eval
`layer_w` is 3 learnable weights under a softmax. The int8 features are **dequantised** to fp32 inside the model using the per-channel `scale` buffer. They start out equal and training decides which layer dominates.

In [ ]:
class WeightedSumHead(nn.Module):
    def __init__(self, scale, n_ws=N_WS, dim=HID, vocab_size=VOCAB_SIZE):
        super().__init__()
        self.layer_w = nn.Parameter(torch.zeros(n_ws))                    # start out equal
        self.register_buffer("scale",
            torch.as_tensor(np.asarray(scale), dtype=torch.float32).view(1, 1, n_ws, dim))
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ELU(), nn.Linear(dim, vocab_size))
    def forward(self, x):                        # x: [B, T, N_WS, 768] int8
        x = x.float() * self.scale               # dequant -> fp32
        w = self.layer_w.softmax(0)
        feat = (x * w[None, None, :, None]).sum(2)   # [B, T, 768]
        return self.net(feat)

def greedy_decode(ids):
    out, prev = [], -1
    for t in ids:
        if t != prev and t != BLANK_ID and t != UNK_ID:
            out.append(ID2CH.get(t, ""))
        prev = t
    return "".join(out).replace("|", " ").strip()

@torch.no_grad()
def evaluate_wer(model, dl, ds):
    model.eval()
    hyps, refs = [], []
    for x, y, xlen, ylen, idxs in dl:
        logits = model(x.to(DEVICE, non_blocking=True))
        pred = logits.argmax(-1).cpu().numpy()
        for b, i in enumerate(idxs):
            hyps.append(greedy_decode(pred[b, :xlen[b]].tolist()))
            refs.append(ds.texts[i])
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps), hyps, refs

## 7. Training
Every epoch prints the **layer weights** (`w=[...]`) so you can see which layer became dominant. The best model is saved to Drive and early stopping is enabled.

In [ ]:
model = WeightedSumHead(FEAT_SCALE).to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
print(f"[MODEL] weighted-sum head, trainable params: {n_par:,}  (layers={WS_LAYERS})")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)
ctc = nn.CTCLoss(blank=BLANK_ID, reduction="mean", zero_infinity=True)

best_cer, best_epoch = float("inf"), 0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0, tot_loss, nb = time.perf_counter(), 0.0, 0
    for x, y, xlen, ylen, idxs in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        logp = logits.log_softmax(-1).transpose(0, 1)
        loss = ctc(logp, y, xlen.to(DEVICE), ylen.to(DEVICE))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item(); nb += 1

    va_wer, va_cer, _, _ = evaluate_wer(model, dev_dl, dev_ds)
    lr_now = optimizer.param_groups[0]["lr"]
    wnow = model.layer_w.softmax(0).detach().cpu().numpy().round(3)
    print(f"epoch {epoch:>3} | loss {tot_loss/nb:.3f} | {time.perf_counter()-t0:.1f}s "
          f"| VAL cer {va_cer*100:.1f}% wer {va_wer*100:.1f}% | lr {lr_now:.1e} | w={wnow}")

    if epoch % 10 == 0:
        tr_wer, tr_cer, _, _ = evaluate_wer(model, train_dl, train_ds)
        print(f"   >>> TRAIN cer {tr_cer*100:.1f}% wer {tr_wer*100:.1f}%")

    scheduler.step(va_cer)
    if va_cer < best_cer:
        best_cer, best_epoch = va_cer, epoch
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "ctc_head_ws.pt"))
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"   [SAVE] new best val-CER {va_cer*100:.1f}% -> {OUTPUT_DIR}/ctc_head_ws.pt")
    elif epoch - best_epoch >= STOP_PATIENCE:
        print(f"[STOP] no improvement for {STOP_PATIENCE} epochs (best {best_cer*100:.1f}% @ e{best_epoch})")
        break

print(f"[DONE] best val-CER {best_cer*100:.1f}% @ epoch {best_epoch}")
print(f"[COMPARE] baseline (layer 9 only) = 8.2% val-CER. The weighted-sum difference is above.")